## Give a website link to generate latest congress member's equity transaction

In [2]:
import os
import sys

sys.path.append(os.path.abspath(".."))
from dotenv import load_dotenv
from scraper import fetch_website_contents  
from openai import OpenAI
import gradio as gr

In [20]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
google_api_key = os.getenv('GEMINI_API_KEY')


if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:8]}")
else:
    print("Google API Key not set")

OpenAI API Key exists and begins sk-proj-
Google API Key exists and begins AIzaSyCr


In [21]:
openai = OpenAI()

gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"

gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)

In [25]:
system_prompt = "You are my personal financial assistant that analyzes congress trading within a website"
user_prompt = """
Here are the contents of a website.
Provides a short summary in table form. 
I want the data from the past one month, 
the table should show who bought what company, show the recent trades first"
"""

# Step 2: Make the messages list
def stream_openai(website):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt + website}
    ] 
    stream = openai.chat.completions.create(
        model='gpt-4.1-mini',
        messages=messages,
        stream=True
    )
    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result

def stream_gemini(website):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt + website}
    ]
    stream = gemini.chat.completions.create(
        model = 'gemini-3.1-flash-lite-preview',
        messages = messages,
        stream = True
    )
    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result


In [26]:
def stream_model(website, model):
    if model=="GPT":
        result = stream_openai(website)
    elif model=="Gemini":
        result = stream_gemini(website)
    else:
        raise ValueError("Unknown model")
    yield from result

In [24]:
system_message = "You are a helpful assistant that responds in markdown without code blocks"

message_input = gr.Textbox(label="Your website:", info="Enter a website for LLM")
model_selector = gr.Dropdown(["GPT", "Gemini"], label="Select model", value="GPT")
message_output = gr.Markdown(label="Response:")

website = fetch_website_contents("https://www.capitoltrades.com/trades")

view = gr.Interface(
    fn=stream_model,
    title="LLMs", 
    inputs=[message_input, model_selector], 
    outputs=[message_output], 
    examples=[
            [website, "GPT"],
            [website, "Gemini"],
        ], 
    flagging_mode="never"
    )
view.launch()

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


Traceback (most recent call last):
  File "/Users/iangoh/ai_projects/llm_engineering/.venv/lib/python3.12/site-packages/gradio/queueing.py", line 759, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/iangoh/ai_projects/llm_engineering/.venv/lib/python3.12/site-packages/gradio/route_utils.py", line 354, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/iangoh/ai_projects/llm_engineering/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 2116, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/iangoh/ai_projects/llm_engineering/.venv/lib/python3.12/site-packages/gradio/blocks.py", line 1635, in call_function
    prediction = await utils.async_iteration(iterator)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/iangoh/ai_projects/llm_engi